### RAG Pipeline - Data Ingestion to vector DB Pipel

In [7]:
import os
from langchain_community.document_loaders import PyPDFLoader,PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

In [ ]:
### Read all the pdf's inside the directory
def process_all_pdf(pdf_directory):
    """Process all pdf files in a directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)

    ###find all PDF files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))

    print(f"Found {len(pdf_files)} PDF files to process")

    for pdf_file in pdf_files:
        print(f"\nProcessing : {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()

            #Add source information to metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'

            all_documents.extend(documents)
            print(f" Loaded {len(documents)} pages")
        except Exception as e:
            print(f" Error : {e}")

    print(f"\nTotal documents loaded : {len(all_documents)}")
    return all_documents

#process all PDFs in the data directory
all_pdf_documents = process_all_pdf("../data")

Found 5PDF files to process

Processing : Examination Guidelines - HWI Qualifiers.pdf
 Loaded 3 pages

Processing : HackWithInfy Practice Questions 2026 Part 1.pdf
 Loaded 7 pages

Processing : HackWithInfy Practice Questions 2026 Part 2.pdf
 Loaded 6 pages

Processing : HackWithInfy Practice Questions 2026 Part 3.pdf


fontTools is required to fully parse the encoding of a CFF Type1 font in font dictionary {'/BaseFont': '/ZHCPFZ+MyriadPro-Regular', '/Encoding': '/WinAnsiEncoding', '/FirstChar': 32, '/FontDescriptor': IndirectObject(75, 0, 2619270034496), '/LastChar': 169, '/Subtype': '/Type1', '/Type': '/Font', '/Widths': [212, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 207, 0, 207, 343, 513, 0, 513, 0, 0, 513, 0, 0, 0, 0, 207, 207, 0, 0, 0, 0, 737, 612, 542, 580, 0, 492, 487, 0, 0, 239, 0, 0, 472, 0, 658, 0, 0, 0, 538, 493, 0, 0, 0, 0, 0, 541, 0, 0, 0, 0, 0, 0, 0, 482, 569, 448, 564, 501, 292, 559, 555, 234, 243, 469, 236, 834, 555, 549, 569, 0, 327, 396, 331, 551, 481, 736, 463, 471, 0, 0, 239, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 677]}, but is not installed. Consider installing fontTools if you encounter encoding problems.


 Loaded 27 pages

Processing : Unit-1 CD CS-603 .pdf
 Loaded 39 pages

Total documents loaded : 82


In [9]:
all_pdf_documents

[Document(metadata={'producer': 'Microsoft® Word for Microsoft 365', 'creator': 'Microsoft® Word for Microsoft 365', 'creationdate': '2026-03-16T11:52:31+05:30', 'msip_label_a0819fa7-4367-4500-ba88-dd630d977609_siteid': '63ce7d59-2f3e-42cd-a8cc-be764cff5eb6', 'msip_label_a0819fa7-4367-4500-ba88-dd630d977609_method': 'Standard', 'msip_label_a0819fa7-4367-4500-ba88-dd630d977609_enabled': 'True', 'author': 'Nabha Belsare', 'moddate': '2026-03-16T11:52:31+05:30', 'source': '..\\data\\pdf\\Examination Guidelines - HWI Qualifiers.pdf', 'total_pages': 3, 'page': 0, 'page_label': '1', 'source_file': 'Examination Guidelines - HWI Qualifiers.pdf', 'file_type': 'pdf'}, page_content='EXAMINATION GUIDELINES FOR HACKWITHINFY QUALIFIERS \nSYSTEM REQUIREMENTS \nPlease ensure that the laptop/desktop computer you plan to use for taking the online test \nfulfills the following system requirements. Taking the test on a mobile or a tablet is not \nrecommended. \n• i3 Core processor or higher  \n• 4GB RAM  

In [11]:
### Text splitting get into chunks

def split_documents(document,chunk_size =1000,chunk_overlap=200):
    """Split documents into smaller chunks for better RAG performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size = chunk_size,
        chunk_overlap = chunk_overlap,
        length_function = len,
        separators =["\n\n","\n"," ",""]
    )
    split_docs=text_splitter.split_documents(document)
    print(f"Split {len(document)} documents into {len(split_docs)} chunks")

    #show example of a chunk
    if split_docs:
        print(f"\nExample chunk :")
        print(f"Content : {split_docs[0].page_content[:200]}...")
        print(f"Metadata : {split_docs[0].metadata}")

    return split_docs

In [12]:
chunks = split_documents(all_pdf_documents)

Split 82 documents into 134 chunks

Example chunk :
Content : EXAMINATION GUIDELINES FOR HACKWITHINFY QUALIFIERS 
SYSTEM REQUIREMENTS 
Please ensure that the laptop/desktop computer you plan to use for taking the online test 
fulfills the following system requir...
Metadata : {'producer': 'Microsoft® Word for Microsoft 365', 'creator': 'Microsoft® Word for Microsoft 365', 'creationdate': '2026-03-16T11:52:31+05:30', 'msip_label_a0819fa7-4367-4500-ba88-dd630d977609_siteid': '63ce7d59-2f3e-42cd-a8cc-be764cff5eb6', 'msip_label_a0819fa7-4367-4500-ba88-dd630d977609_method': 'Standard', 'msip_label_a0819fa7-4367-4500-ba88-dd630d977609_enabled': 'True', 'author': 'Nabha Belsare', 'moddate': '2026-03-16T11:52:31+05:30', 'source': '..\\data\\pdf\\Examination Guidelines - HWI Qualifiers.pdf', 'total_pages': 3, 'page': 0, 'page_label': '1', 'source_file': 'Examination Guidelines - HWI Qualifiers.pdf', 'file_type': 'pdf'}
